# Brazil Mid-Market Deals Radar

A small, reproducible tracker of **Brazilian mid-market deal flow** — announced
M&A, private equity, venture, and select debt — read through a credit lens.
Everything runs top-to-bottom in this notebook: generate a synthetic demo
dataset, validate it, and profile the **most active acquirers** over a trailing
12-month window, with a scaffold for a later **public** credit-stress overlay.

The value on show is the **framework**: a clean data model, validation,
acquirer-name canonicalization, and a reproducible analysis — not statistical
breadth. I approach this as a **deals generalist who brings a sharper credit lens
to diligence**, treating leverage and credit quality as part of how you read a
transaction rather than as a distressed-only specialty.

> **Portfolio / practice build.** The dataset here is **synthetic**, generated
> below with a fixed seed. Every row carries `source = SYNTHETIC-EXAMPLE` and uses
> fictional placeholder names, so no real transaction is implied. To use this for
> real, replace the synthetic rows and log actual deals from **public** sources
> via the `add_deal(...)` helper near the end.

*Just run all cells.* Requirements: `pandas`, `matplotlib`
(`pip install pandas matplotlib`).

## 0 · Configuration

In [ ]:
import csv, datetime as dt, difflib, os, random
import pandas as pd
import matplotlib.pyplot as plt

# Editable knobs
SEED         = 42      # reproducible synthetic draw
N_DEALS      = 55      # how many synthetic deals to create
TOP_N        = 10      # acquirers shown in each headline chart
WINDOW_DAYS  = 365     # trailing coverage window, anchored on the latest deal
DEALS_FILE   = "deals.csv"
FIG_DIR      = "figures"

# If deals.csv already exists, we reuse it (so you don't overwrite real entries).
# Flip this to True to always regenerate the synthetic demo dataset.
FORCE_REGENERATE = False

os.makedirs(FIG_DIR, exist_ok=True)

# Schema (column order is used verbatim by the tooling)
COLUMNS = ["deal_id","announce_date","target","acquirer_raw","acquirer_canonical",
           "acquirer_type","deal_type","sector","value_brl_mm","stake_pct",
           "source","source_url","notes"]
ACQUIRER_TYPES = ["PE","Estrategico","VC","FO"]
DEAL_TYPES     = ["M&A","PE","VC","divida","distressed"]
REQUIRED = ["deal_id","announce_date","target","acquirer_raw","acquirer_canonical",
            "acquirer_type","deal_type","sector","source","source_url"]

# Editable sector-normalization map (messy label -> tidy label).
# Unmapped sectors pass through unchanged, so nothing is silently dropped.
SECTOR_MAP = {
    "transporte e logistica":"Transporte & Logística","logistica":"Transporte & Logística",
    "alimentos e bebidas":"Alimentos & Bebidas","alimentos":"Alimentos & Bebidas",
    "saude":"Saúde","tecnologia":"Tecnologia","ti":"Tecnologia",
    "agronegocio":"Agronegócio","agro":"Agronegócio","varejo":"Varejo","energia":"Energia",
    "servicos":"Serviços","servicos financeiros":"Serviços Financeiros",
    "educacao":"Educação","construcao":"Construção","industria":"Indústria",
}

BAR = "#3b5b7d"
plt.rcParams.update({"figure.dpi":110,"axes.spines.top":False,
                     "axes.spines.right":False,"font.size":11})

## 1 · Synthetic demo data

A seeded generator fabricates clearly-labeled sample deals so the charts have
something to show. This is **demo data, not collection** — there is nothing real
here to gather. Fictional acquirers use placeholder (Greek-letter) names; a few
carry spelling variants on purpose, to exercise the canonicalization step later.

In [ ]:
# (canonical, type, [raw spelling variants], activity weight)
_ACQUIRERS = [
    ("Alpha Capital Partners","PE",["Alpha Capital Partners","Alpha Capital","Alpha Cap. Partners"],9),
    ("Beta Equity","PE",["Beta Equity","Beta Equity Ltda"],7),
    ("Gamma Capital","PE",["Gamma Capital"],5),
    ("Sigma Investimentos","PE",["Sigma Investimentos"],4),
    ("Omega Partners","PE",["Omega Partners","Ômega Partners"],4),
    ("Tau Capital","PE",["Tau Capital"],2),
    ("Grupo Delta","Estrategico",["Grupo Delta","Grupo Delta S.A."],6),
    ("Epsilon Industria","Estrategico",["Epsilon Industria","Épsilon Indústria"],4),
    ("Holding Zeta","Estrategico",["Holding Zeta"],3),
    ("Grupo Ni","Estrategico",["Grupo Ni","Grupo Ni Holding"],2),
    ("Kappa Ventures","VC",["Kappa Ventures"],5),
    ("Lambda Ventures","VC",["Lambda Ventures"],4),
    ("Theta Ventures","VC",["Theta Ventures"],3),
    ("Phi Ventures","VC",["Phi Ventures"],2),
    ("Family Office Ikaros","FO",["Family Office Ikaros","FO Ikaros"],3),
    ("Escritorio Mu","FO",["Escritorio Mu"],2),
]
_SECTORS = ["Agronegocio","Saude","Tecnologia","Varejo","Transporte e logistica",
            "Alimentos e bebidas","Energia","Servicos financeiros","Educacao",
            "Construcao","Industria","Servicos"]
_ROOTS = ["Aurora","Bandeira","Cerrado","Dunas","Estrela","Farol","Guara","Horizonte",
          "Ipe","Jacaranda","Litoral","Marimba","Norte","Oceano","Palmares","Quartzo",
          "Raizes","Serrano","Tucano","Uniao","Verde","Xingu","Zenite","Buriti"]
_SUFFIX = {"Agronegocio":"Agro","Saude":"Saude","Tecnologia":"Tech","Varejo":"Varejo",
           "Transporte e logistica":"Log","Alimentos e bebidas":"Foods","Energia":"Energia",
           "Servicos financeiros":"Fin","Educacao":"Educa","Construcao":"Engenharia",
           "Industria":"Industrial","Servicos":"Servicos"}
_NOTES = ["Amostra sintetica. Plataforma de buy-and-build; observar alavancagem pos-deal.",
          "Amostra sintetica. Consolidacao setorial; multiplo aparentemente esticado.",
          "Amostra sintetica. Rodada minoritaria; sem controle.",
          "Amostra sintetica. Credito privado; foco em geracao de caixa do alvo.",
          "Amostra sintetica. Situacao especial; tese de turnaround.",
          "Amostra sintetica."]

def _wpick(rng, items, weights):
    return rng.choices(items, weights=weights, k=1)[0]

def _deal_type(rng, atype):
    w = {"VC":[10,8,70,2,2],"Estrategico":[70,8,5,7,5],
         "FO":[30,15,10,30,10]}.get(atype,[45,30,5,8,12])  # PE default
    return _wpick(rng, DEAL_TYPES, w)

def _value(rng, deal_type):
    if rng.random() > 0.65:
        return ""                          # ~35% undisclosed
    base = max(12, min(rng.lognormvariate(4.8, 0.7), 820))
    if deal_type == "VC":
        base = max(8, base * 0.35)
    return str(int(round(base,-1)) if base >= 100 else int(round(base)))

def _stake(rng, deal_type):
    if deal_type == "divida": return ""
    if deal_type == "VC":     return str(rng.randint(5,30))
    if deal_type in ("M&A","distressed"): return str(rng.choice([100,100,100,70,60,51,80]))
    return str(rng.choice([100,80,70,60,51,40]))

def generate_deals(n=N_DEALS, seed=SEED):
    rng = random.Random(seed)
    names   = [a[0] for a in _ACQUIRERS]
    weights = [a[3] for a in _ACQUIRERS]
    start, end = dt.date(2025,7,1), dt.date(2026,6,25)
    span = (end - start).days
    rows = []
    for _ in range(n):
        canon = _wpick(rng, names, weights)
        _, atype, variants, _ = next(a for a in _ACQUIRERS if a[0] == canon)
        sector = rng.choice(_SECTORS)
        dtp    = _deal_type(rng, atype)
        val    = _value(rng, dtp)
        rows.append({
            "announce_date": (start + dt.timedelta(days=rng.randint(0, span))).isoformat(),
            "target": f"{rng.choice(_ROOTS)} {_SUFFIX[sector]}",
            "acquirer_raw": rng.choice(variants), "acquirer_canonical": canon,
            "acquirer_type": atype, "deal_type": dtp, "sector": sector,
            "value_brl_mm": val, "stake_pct": _stake(rng, dtp),
            "source": "SYNTHETIC-EXAMPLE",
            "notes": ("Amostra sintetica. Valor nao divulgado pelas partes."
                      if val == "" else rng.choice(_NOTES)),
        })
    rows.sort(key=lambda r: r["announce_date"])
    for i, r in enumerate(rows, 1):
        r["deal_id"] = str(i); r["source_url"] = f"https://example.com/synthetic/{i}"
    return rows

def write_csv(rows, path=DEALS_FILE):
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=COLUMNS); w.writeheader()
        for r in rows: w.writerow({c: r.get(c,"") for c in COLUMNS})

if FORCE_REGENERATE or not os.path.exists(DEALS_FILE):
    write_csv(generate_deals())
    print(f"Generated {N_DEALS} synthetic deals -> {DEALS_FILE} (seed={SEED}).")
else:
    print(f"Using existing {DEALS_FILE} (set FORCE_REGENERATE=True to overwrite).")

## 2 · Validate

The same integrity checks the CLI ran: schema, types, ranges, unique/sequential ids.

In [ ]:
def _iso(v):
    try: dt.date.fromisoformat(v); return True
    except (ValueError, TypeError): return False
def _blank(v): return v is None or str(v).strip() == ""
def _num(v, lo=None, hi=None):
    try: n = float(v)
    except (ValueError, TypeError): return False
    return (lo is None or n >= lo) and (hi is None or n <= hi)

def validate(path=DEALS_FILE):
    problems = []
    with open(path, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh); header = reader.fieldnames or []
        rows = [dict(r) for r in reader]
    if header != COLUMNS:
        return [f"Header mismatch: {header}"]
    seen = {}
    for i, row in enumerate(rows, 1):
        loc = f"row {i} (deal_id={row.get('deal_id','?')})"
        for f in REQUIRED:
            if _blank(row.get(f)): problems.append(f"{loc}: '{f}' blank.")
        did = row.get("deal_id","")
        if not _blank(did):
            if not did.isdigit(): problems.append(f"{loc}: deal_id not integer.")
            elif did in seen: problems.append(f"{loc}: duplicate deal_id.")
            else: seen[did] = i
        if not _blank(row.get("announce_date")) and not _iso(row["announce_date"]):
            problems.append(f"{loc}: bad announce_date.")
        if row.get("acquirer_type") not in ACQUIRER_TYPES:
            problems.append(f"{loc}: bad acquirer_type.")
        if row.get("deal_type") not in DEAL_TYPES:
            problems.append(f"{loc}: bad deal_type.")
        if not _blank(row.get("value_brl_mm")) and not _num(row["value_brl_mm"], lo=0):
            problems.append(f"{loc}: bad value_brl_mm.")
        if not _blank(row.get("stake_pct")) and not _num(row["stake_pct"], lo=0, hi=100):
            problems.append(f"{loc}: bad stake_pct.")
    ids = [int(r["deal_id"]) for r in rows if r["deal_id"].isdigit()]
    if len(ids) == len(rows) and sorted(ids) != list(range(1, len(rows)+1)):
        problems.append("deal_id not a clean 1..N run.")
    return problems

issues = validate()
print("OK — validation passed." if not issues else f"FAIL — {len(issues)} issue(s):")
for p in issues: print("  -", p)
assert not issues, "Fix the dataset before continuing."

## 3 · Load & clean

Parse dates and numerics, normalize sectors, spot-check the acquirer
canonicalization, and restrict to the trailing-12-month window anchored on the
**latest** `announce_date`. All acquirer analysis groups on `acquirer_canonical`.

In [ ]:
df = pd.read_csv(DEALS_FILE, dtype=str).fillna("")
df["announce_date"] = pd.to_datetime(df["announce_date"], errors="coerce")
df["value_brl_mm"]  = pd.to_numeric(df["value_brl_mm"], errors="coerce")   # blank -> NaN
df["stake_pct"]     = pd.to_numeric(df["stake_pct"], errors="coerce")
df["sector_norm"]   = df["sector"].map(lambda s: SECTOR_MAP.get(str(s).strip().lower(), str(s).strip()))

print("acquirer_canonical  <-  distinct acquirer_raw spellings folded in:")
for canon, raws in df.groupby("acquirer_canonical")["acquirer_raw"].agg(lambda s: sorted(set(s))).items():
    flag = "  <-- multiple spellings" if len(raws) > 1 else ""
    print(f"  {canon}: {raws}{flag}")

anchor = df["announce_date"].max()
start  = anchor - pd.Timedelta(days=WINDOW_DAYS)
window = df[df["announce_date"].between(start, anchor)].copy()
print(f"\nCoverage window: {start.date()} -> {anchor.date()}  ({len(window)} of {len(df)} rows).")
df.head()

## 4 · Most active acquirers — by deal count

In [ ]:
by_count = window.groupby("acquirer_canonical").size().sort_values().tail(TOP_N)
fig, ax = plt.subplots(figsize=(8, max(2.5, 0.5*len(by_count)+1)))
bars = ax.barh(by_count.index, by_count.values, color=BAR)
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_xlabel("Number of announced deals")
ax.set_title(f"Most active acquirers by deal count\ntrailing 12 months to "
             f"{anchor.date()} · n={len(window)} deals", loc="left")
ax.margins(x=0.12); fig.tight_layout()
fig.savefig(f"{FIG_DIR}/acquirers_by_count.png", bbox_inches="tight")
plt.show()

## 5 · Most active acquirers — by disclosed value

Undisclosed values are excluded, and the excluded count is reported.

In [ ]:
disclosed = window[window["value_brl_mm"].notna()]
n_excl = len(window) - len(disclosed)
by_value = disclosed.groupby("acquirer_canonical")["value_brl_mm"].sum().sort_values().tail(TOP_N)
fig, ax = plt.subplots(figsize=(8, max(2.5, 0.5*len(by_value)+1)))
bars = ax.barh(by_value.index, by_value.values, color=BAR)
ax.bar_label(bars, fmt="%.0f", padding=3, fontsize=10)
ax.set_xlabel("Aggregate disclosed value (R$ mm)")
ax.set_title(f"Most active acquirers by disclosed value\ntrailing 12 months to "
             f"{anchor.date()} · {n_excl} of {len(window)} deals undisclosed (excluded)", loc="left")
ax.margins(x=0.14); fig.tight_layout()
fig.savefig(f"{FIG_DIR}/acquirers_by_value.png", bbox_inches="tight")
plt.show()
print(f"{n_excl} of {len(window)} deals had undisclosed value and are excluded from the value chart.")

## 6 · Profile cuts

Descriptive slices by acquirer type, deal type, and normalized sector.

In [ ]:
def cut(col, label):
    t = (window.groupby(col)
               .agg(deals=("deal_id","size"), disclosed_value_brl_mm=("value_brl_mm","sum"))
               .sort_values("deals", ascending=False))
    print(f"— By {label} —"); print(t.to_string()); print()

cut("acquirer_type", "acquirer type")
cut("deal_type", "deal type")
cut("sector_norm", "sector (normalized)")

## 7 · Log a real deal *(optional)*

To add an actual, publicly-sourced deal, call `add_deal(...)` in a cell. It
assigns the next `deal_id`, suggests a canonical acquirer name (fuzzy-matched
against existing ones to avoid near-duplicates), appends to `deals.csv`, and
re-validates. Manual entry only — nothing is fetched.

In [ ]:
def suggest_canonical(raw, existing):
    m = difflib.get_close_matches(raw, list(existing), n=1, cutoff=0.6)
    return m[0] if m else raw

def add_deal(announce_date, target, acquirer_raw, acquirer_type, deal_type, sector,
             source, source_url, value_brl_mm="", stake_pct="", notes="",
             acquirer_canonical=None, path=DEALS_FILE):
    with open(path, newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    next_id = max((int(r["deal_id"]) for r in rows if r["deal_id"].isdigit()), default=0) + 1
    existing = {r["acquirer_canonical"] for r in rows if r.get("acquirer_canonical")}
    canon = acquirer_canonical or suggest_canonical(acquirer_raw, existing)
    rows.append({"deal_id":str(next_id),"announce_date":announce_date,"target":target,
                 "acquirer_raw":acquirer_raw,"acquirer_canonical":canon,
                 "acquirer_type":acquirer_type,"deal_type":deal_type,"sector":sector,
                 "value_brl_mm":str(value_brl_mm),"stake_pct":str(stake_pct),
                 "source":source,"source_url":source_url,"notes":notes})
    write_csv(rows, path)
    issues = validate(path)
    print(f"Added deal_id {next_id} (acquirer_canonical='{canon}'). "
          + ("Validation OK." if not issues else f"WARNING: {len(issues)} issue(s)."))
    return next_id

# Example (uncomment, edit with a real public deal, and re-run cells 3-6):
# add_deal("2026-06-30", "Real Target Co", "Some Fund", "PE", "M&A", "Tecnologia",
#          "Brazil Journal", "https://braziljournal.com/...", value_brl_mm=150, stake_pct=100,
#          notes="Public source; credit note here.")
print("add_deal() ready. Log real deals from public sources only.")

## 8 · Phase 2 — credit x deals overlay

The thesis: **where deal activity is high in a sector whose credit is deteriorating,
there is usually a story worth digging into** - opportunity and risk that most people
are not crossing.

The **per-sector credit stress index** is built from one public source: the Brazilian
Central Bank's **SCR.data**, which publishes the active loan book and non-performing
loans broken down by CNAE section for corporate borrowers. Two signals come from it:

- **Level** - the sector's current NPL (over 90 days past due / active book).
- **12-month trend** - the change in that NPL, in percentage points, over a year.

They are combined via z-score, with the **trend weighted heavier than the level**
(0.6 vs 0.4): a book already deteriorating is more informative than one that is high
but stable.

**Why SCR.data and not SGS.** SGS - the BCB source most people reach for - has no
sectoral corporate NPL series. Its corporate NPL series are broken down by loan
*modality* (vehicles, credit cards, working capital), not by economic activity. The
sectoral cut exists only in SCR.data.

**Reproducibility.** The next cell downloads the annual SCR.data archives, aggregates
CNAE sections into this radar's sectors - weighting by the loan book rather than
averaging rates - and writes `stress_indicators.csv`, stamped with the SHA-256 of the
source files. That CSV is committed, so a fresh clone runs offline with no download.
Set `REFRESH_STRESS = True` to rebuild it.

In [ ]:
# --- Credit stress: load the committed CSV, or rebuild it from the BCB -----
REFRESH_STRESS = False          # True -> download from the BCB and rewrite the CSV
SCR_DATA_BASE  = "2026-03"      # most recent SCR.data reference month (YYYY-MM)

import hashlib, io, time, unicodedata, zipfile
from datetime import date
from pathlib import Path
import requests

SCR_URL   = "https://www.bcb.gov.br/pda/desig/scrdata_{year}.zip"
STRESS_FILE = "stress_indicators.csv"

# The NPL numerator. The BCB defines non-performance as the FULL balance of loans
# with any instalment more than 90 days overdue - not merely the overdue instalments
# themselves. `carteira_inadimplencia` is that full balance; `vencido_acima_de_90_dias`
# is only the past-due portion and understates NPL by roughly half. SCR.data also
# offers `ativo_problematico`, a broader measure that adds restructurings and other
# impaired exposures; swap it in below to run the index on that definition instead.
NPL_FIELD = "carteira_inadimplencia"
SCR_SOURCE  = ("BCB SCR.data (v2) - carteira_inadimplencia / carteira_ativa "
               "by CNAE section, corporate borrowers")
SCR_DOC     = "https://dadosabertos.bcb.gov.br/dataset/scr_data"

# NOTE: add  .scr_cache/  to .gitignore - those archives are ~250 MB.

# CNAE section -> this radar's sector labels (they must match SECTOR_MAP's output).
# The SCR field carries the section NAME, not its letter. Sectors spanning several
# sections sum book and past-due BEFORE dividing: a book-weighted rate, not an
# average of rates.
CNAE_TO_SECTOR = {
    "agricultura":                   "Agronegócio",
    "industrias de transformacao":   "Indústria",
    "eletricidade e gas":            "Energia",
    "construcao":                    "Construção",
    "comercio":                      "Varejo",
    "transporte":                    "Transporte & Logística",
    "informacao e comunicacao":      "Tecnologia",
    "atividades financeiras":        "Serviços Financeiros",
    "educacao":                      "Educação",
    "saude humana":                  "Saúde",
    "atividades profissionais":      "Serviços",
    "atividades administrativas":    "Serviços",
    "outras atividades de servicos": "Serviços",
}
# Sections deliberately left out (no matching deals): public administration, mining,
# utilities/sanitation, hospitality, real estate, arts, international bodies, and
# "not reported".


def _sector_of(section_name):
    n = unicodedata.normalize("NFKD", str(section_name))
    n = "".join(ch for ch in n if not unicodedata.combining(ch)).strip().lower()
    for key, sector in CNAE_TO_SECTOR.items():
        if n.startswith(key):
            return sector
    return None


SCR_CACHE = Path(".scr_cache")     # ZIPs baixados ficam aqui. NÃO comitar.


def _download_zip(year, tries=4):
    """Stream the annual archive to disk, with retries. Big files drop mid-flight."""
    SCR_CACHE.mkdir(exist_ok=True)
    path = SCR_CACHE / f"scrdata_{year}.zip"

    # Already have a complete, valid archive? Reuse it.
    if path.exists() and zipfile.is_zipfile(path):
        print(f"  cached {path} ({path.stat().st_size/1e6:.1f} MB)")
        return path

    url = SCR_URL.format(year=year)
    for attempt in range(1, tries + 1):
        try:
            print(f"  downloading {url} (attempt {attempt}/{tries})")
            with requests.get(url, stream=True, timeout=(30, 300)) as r:
                if r.status_code != 200:
                    raise RuntimeError(f"BCB returned HTTP {r.status_code}")
                expected = int(r.headers.get("Content-Length", 0))
                got = 0
                tmp = path.with_suffix(".part")
                with open(tmp, "wb") as fh:
                    for block in r.iter_content(chunk_size=1 << 20):   # 1 MB
                        fh.write(block)
                        got += len(block)
                        if expected:
                            print(f"\r    {got/1e6:6.1f} / {expected/1e6:.1f} MB",
                                  end="", flush=True)
                print()

            if expected and got < expected:
                raise IOError(f"truncated: {got} of {expected} bytes")
            if not zipfile.is_zipfile(tmp):
                raise IOError("downloaded file is not a valid ZIP")

            tmp.replace(path)
            print(f"    ok - {path.stat().st_size/1e6:.1f} MB")
            return path

        except Exception as e:
            print(f"    failed: {type(e).__name__}: {e}")
            if attempt == tries:
                raise RuntimeError(
                    f"could not download {url} after {tries} attempts. "
                    f"Nothing was written."
                )
            time.sleep(3 * attempt)


def _scr_npl(year, month):
    """Fetch one annual archive, read one month, return NPL % by sector."""
    path = _download_zip(year)
    sha = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
    print(f"    sha256[:16] = {sha}")

    zf  = zipfile.ZipFile(path)
    tag = f"{year}{month:02d}"
    hits = [n for n in zf.namelist() if tag in n and n.lower().endswith(".csv")]
    if not hits:
        raise RuntimeError(f"no CSV for {tag} in the {year} archive - not published yet?")

    book, past_due, tot_b, tot_v = {}, {}, 0.0, 0.0
    with zf.open(hits[0]) as fh:
        for chunk in pd.read_csv(fh, sep=";", decimal=",", chunksize=500_000,
                                 low_memory=False, encoding="utf-8-sig"):
            pj = chunk[chunk["cliente"].astype(str).str.upper().str.startswith("PJ")]
            if pj.empty:
                continue
            g = pd.DataFrame({
                "sector": pj["cnae_ocupacao"].map(_sector_of),
                "b": pd.to_numeric(pj["carteira_ativa"], errors="coerce"),
                "v": pd.to_numeric(pj[NPL_FIELD], errors="coerce"),
            }).dropna(subset=["b", "v"])
            tot_b += g["b"].sum()
            tot_v += g["v"].sum()
            for s, sub in g.dropna(subset=["sector"]).groupby("sector"):
                book[s]     = book.get(s, 0.0) + sub["b"].sum()
                past_due[s] = past_due.get(s, 0.0) + sub["v"].sum()

    coverage = 100 * sum(book.values()) / tot_b if tot_b else 0
    overall  = 100 * tot_v / tot_b if tot_b else 0
    print(f"    {len(book)} sectors | {coverage:.0f}% of the corporate book "
          f"| overall corporate NPL {overall:.2f}%")

    # Sanity gates. The dangerous failure here is not a crash - it is a number that
    # looks plausible and is wrong. An earlier version of this mapping silently
    # collapsed Retail, Construction and Industry into one bucket and drew a perfectly
    # clean chart from nonsense. These gates exist so that cannot happen quietly.
    if len(book) < 10:
        raise RuntimeError(f"only {len(book)} sectors mapped - the CNAE mapping is broken")
    if coverage < 60:
        raise RuntimeError(f"mapping covers only {coverage:.0f}% of the corporate book")
    # Cross-check against a figure anyone can verify: BCB publishes headline corporate
    # NPL (SGS 21083) in the low single digits. A result far outside that band means the
    # numerator or the filter is wrong - as happened when this used the past-due
    # instalments instead of the full impaired balance, and printed 1.25%.
    if not 1.5 <= overall <= 8:
        raise RuntimeError(
            f"overall corporate NPL of {overall:.2f}% is outside the plausible band "
            f"(BCB headline sits in the low single digits). Check NPL_FIELD.")

    return pd.Series({s: 100 * past_due[s] / book[s] for s in book}).sort_index(), sha


if REFRESH_STRESS or not Path(STRESS_FILE).exists():
    yr, mo = map(int, SCR_DATA_BASE.split("-"))
    print(f"Rebuilding the stress index from BCB SCR.data "
          f"(level {yr}-{mo:02d}, trend vs {yr-1}-{mo:02d})\n")
    now,  sha_now  = _scr_npl(yr, mo)
    prev, sha_prev = _scr_npl(yr - 1, mo)

    both = now.index.intersection(prev.index)
    if len(both) < 10:
        raise RuntimeError(f"only {len(both)} sectors present in both months")

    stress = pd.DataFrame({
        "sector": both,
        "npl_level_pct": now[both].round(3).values,
        "npl_trend_12m_pp": (now[both] - prev[both]).round(3).values,
        "source": SCR_SOURCE,
        "reference_period": f"{yr-1}-{mo:02d} -> {yr}-{mo:02d}",
        "source_url": SCR_DOC,
        "source_sha256": f"{sha_prev} / {sha_now}",
        "retrieved_at": date.today().isoformat(),
    })
    stress.to_csv(STRESS_FILE, index=False)      # the only line that writes
    print(f"\nWrote {STRESS_FILE} from real BCB data.")
else:
    stress = pd.read_csv(STRESS_FILE)
    print(f"Loaded {STRESS_FILE} from disk - no download.")
    print("Set REFRESH_STRESS = True to rebuild it from the BCB.")

# Say plainly where the numbers come from. The notebook should never be vague
# about its own provenance.
if "SYNTHETIC" in str(stress["source"].iloc[0]).upper():
    print("\nSTRESS = SYNTHETIC DATA - illustrative only.")
else:
    print(f"\nSTRESS = REAL DATA - BCB SCR.data | "
          f"reference period: {stress['reference_period'].iloc[0]}")


In [ ]:
# --- Per-sector credit stress index ----------------------------------------
W_LEVEL, W_TREND = 0.4, 0.6      # trend weighted heavier than level


def _z(s):
    return (s - s.mean()) / s.std(ddof=0)


stress["z_level"]     = _z(stress["npl_level_pct"])
stress["z_trend"]     = _z(stress["npl_trend_12m_pp"])
stress["stress_index"] = W_LEVEL * stress["z_level"] + W_TREND * stress["z_trend"]

print(f"Credit stress index (weights: level={W_LEVEL}, trend={W_TREND})")
print(f"NPL = {NPL_FIELD} / carteira_ativa, per BCB's definition\n")
print(stress[["sector", "npl_level_pct", "npl_trend_12m_pp", "stress_index"]]
      .sort_values("stress_index", ascending=False)
      .round(2)
      .to_string(index=False))


print(f"\nThe z-score is computed across the {len(stress)} sectors above, not against "
      f"the whole economy: a stress index of 0 means average *within this set*.")

In [ ]:
# --- Overlay: deal activity x credit stress ---------------------------------
# One deal sector has no CNAE section of its own: food & beverage sits inside
# "manufacturing". We borrow manufacturing's stress as an explicit, flagged proxy
# rather than dropping the sector without a word.
STRESS_PROXY = {"Alimentos & Bebidas": "Indústria"}

activity = (window.groupby("sector_norm")
                  .agg(deals=("deal_id", "size"),
                       disclosed_value_brl_mm=("value_brl_mm", "sum"))
                  .reset_index()
                  .rename(columns={"sector_norm": "sector"}))
activity["stress_sector"] = activity["sector"].replace(STRESS_PROXY)

overlay = (activity.merge(stress[["sector", "stress_index", "npl_level_pct",
                                  "npl_trend_12m_pp"]],
                          left_on="stress_sector", right_on="sector",
                          how="inner", suffixes=("", "_stress"))
           .sort_values("stress_index", ascending=False)
           .reset_index(drop=True))

# If the labels ever drift apart, say so loudly instead of drawing an empty chart.
if len(overlay) < 4:
    print("Only", len(overlay), "sectors joined - the labels do not line up.")
    print("  deal sectors  :", sorted(activity["sector"].unique()))
    print("  stress sectors:", sorted(stress["sector"].unique()))
    raise ValueError("Sector join failed. Reconcile SECTOR_MAP with CNAE_TO_SECTOR.")

missing = sorted(set(activity["sector"]) - set(overlay["sector"]))
if missing:
    print(f"No credit stress available for: {', '.join(missing)} "
          f"(no matching CNAE section). Excluded from the chart.\n")

print(overlay[["sector", "deals", "disclosed_value_brl_mm",
               "npl_level_pct", "npl_trend_12m_pp", "stress_index"]]
      .round(2).to_string(index=False))

x, y = overlay["deals"], overlay["stress_index"]
val  = overlay["disclosed_value_brl_mm"].fillna(0)
size = 90 + 620 * (val / val.max() if val.max() > 0 else 0)

fig, ax = plt.subplots(figsize=(9.2, 6.6))
ax.axhline(y.mean(), color="0.78", lw=1, zorder=1)
ax.axvline(x.mean(), color="0.78", lw=1, zorder=1)

# The thesis corner: busy sectors whose credit is above-average stressed.
ax.axhspan(y.mean(), y.max() + 0.7, xmin=0.5, color="#b23a3a", alpha=0.07, zorder=0)

ax.scatter(x, y, s=size, color=BAR, alpha=0.8, edgecolor="white", linewidth=1.2, zorder=3)
# Labels collide when two sectors land close together (Industry and Food & Beverage
# share a stress value by construction). Nudge crowded ones apart.
_placed = []
for _, r in overlay.iterrows():
    label = r["sector"] + ("*" if r["sector"] in STRESS_PROXY else "")
    px, py = r["deals"], r["stress_index"]
    crowded = any(abs(px - qx) <= 1.2 and abs(py - qy) < 0.30 for qx, qy in _placed)
    dy = -20 if crowded else 12
    ax.annotate(label, (px, py), xytext=(0, dy), textcoords="offset points",
                ha="center", va="top" if crowded else "bottom", fontsize=9)
    _placed.append((px, py))

ax.set_xlabel("Announced deals (trailing 12 months)")
ax.set_ylabel("Credit stress index (z-score, 0 = average)")
ax.set_title("Deal activity vs. credit stress by sector\n"
             "top-right = thesis quadrant: active sectors where credit is deteriorating",
             fontsize=11)
if STRESS_PROXY:
    ax.text(0.99, 0.01, "* stress proxied by the parent CNAE section",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8, color="0.45")
ax.margins(0.17)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/deals_vs_stress.png", dpi=140, bbox_inches="tight")
plt.show()

_real = "SYNTHETIC" not in str(stress["source"].iloc[0]).upper()
_ylab = "real BCB data" if _real else "SYNTHETIC (not yet rebuilt from the BCB)"
print(f"\nBubble size = disclosed value. Credit stress (Y) is {_ylab}; deal activity (X) "
      f"is synthetic and labeled. The chart demonstrates the method, not the market.")


## 9 · Limitations

- **The deals are synthetic.** Every deal record is generated and labeled
  (`SYNTHETIC-EXAMPLE`). No public feed of Brazilian mid-market transactions exists to
  draw on, so the deal side demonstrates the framework, not the market. The credit side
  is real.
- **Hybrid axes.** The Phase 2 chart crosses real credit stress (Y) with synthetic deal
  activity (X). What it shows is the *method* of crossing them - the position of any
  given bubble is not a market call.
- **Taxonomy mapping.** CNAE sections do not map 1:1 onto deal sectors. The mapping is
  an explicit approximation; part of the corporate loan book is excluded (public
  administration, mining, real estate, hospitality, and others), and Food & Beverage
  borrows the stress of its parent section, manufacturing.
- **Universe mismatch.** SCR covers every company with registered credit, from micro to
  large. The deals here are mid-market, so a sector's NPL is pulled by its larger firms.
- **Publication lag.** SCR.data is released with a delay. Credit stress is a
  recent-past snapshot, not today.
- **Small reference set.** The z-score is computed across the ~11 mapped sectors, not the full economy. Mean and standard deviation rest on few observations, so a sector's position shifts if the mapping changes. Read the ranking, not the decimals.
- **The index is a simplification.** Two correlated signals and hand-picked weights. It
  orients; it does not settle anything.
- **Acquirer canonicalization is heuristic.** Spelling variants are folded by an
  editable map, not by entity resolution against a registry.